## *** Parallel Chains ***


In [4]:
# Task-1

from langchain_core.prompts import ChatPromptTemplate

primary_prompt = ChatPromptTemplate.from_messages([("system","you are a professional Movie Reviewer"),("human","{movie}")])

In [28]:
# Task-2
import os
import httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

llm_access = ChatOpenAI(model="gpt-5-mini",
temperature=0,
api_key=os.environ["OPENAI_API_KEY"],
#base_url="https://alertmanager.tecnicasreunidas.es/v1",
base_url=f'{os.environ["BASE_URL"]}v1',
http_client=httpx.Client(verify=False)
)

llm_access.invoke("rangasthalam")

AIMessage(content='Do you mean the 2018 Telugu film Rangasthalam? If so, here’s a short overview — tell me what you want next (plot, cast, songs, reviews, where to watch, trivia, etc.).\n\n- Title: Rangasthalam (2018)\n- Director: Sukumar\n- Stars: Ram Charan and Samantha Ruth Prabhu (lead roles)\n- Music: Devi Sri Prasad\n- Cinematography: R. Rathnavelu\n- Premise (brief): A period drama set in a 1980s village called Rangasthalam, focusing on village politics, exploitation by a corrupt local leader, and the fight for justice led by the film’s protagonist, who has a hearing impairment.\n- Reception: Widely praised for performances, direction, music and production design; a major commercial success and critically acclaimed.\n\nWould you like a full plot summary (spoiler warning), a full cast list, soundtrack details, critical reception and awards, or where to stream it?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 984, 'prompt_tokens': 1

In [6]:
#Task-3
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

In [7]:
#Task-4

from langchain_core.runnables import RunnableLambda
def dict_make(text:str)->dict:
    return {"text":text}

runable_dict = RunnableLambda(dict_make)


### Parallel Chain 1

In [26]:
def linkedin_chain(text:dict):


    linkedin_prompt = ChatPromptTemplate.from_messages([("system","you are a LinkedIn Post  Generator"),("human","create a post for follwoing {text}")])
    chain_linkedin =  linkedin_prompt | llm_access | str_parser
    return chain_linkedin

runnable_linkedin_chain = RunnableLambda(linkedin_chain)

#### Parallel Chain 2

In [29]:
def insta_chain(text:dict,):

    instagram_prompt = ChatPromptTemplate.from_messages([("system","you are a Instagram Post  Generator"),("human","create a post for follwoing {text}")])
    chain_insta = instagram_prompt | llm_access | str_parser
    return chain_insta

runanable_instagram_chain = RunnableLambda(insta_chain)

## Final Orchestration

In [30]:
from langchain_core.runnables import RunnableParallel

final_chain = ( primary_prompt | llm_access | str_parser | runable_dict | RunnableParallel(branches = {"linkedin":runnable_linkedin_chain,"instagram":runanable_instagram_chain}))

##### Final chain invocation

In [31]:
final_chain.invoke("Rangasthalam")

{'branches': {'linkedin': 'Rangasthalam (2018) — a sun-baked, uncompromising epic about power, pride and revenge — is a masterclass in rooted storytelling.\n\nWhy it stands out:\n- Sukumar builds a fully lived-in 1980s Rayalaseema village: smells, rhythms and grudges that feel lived-in rather than staged.\n- Ram Charan sheds star-glow as Chitti Babu: physical, quiet and unpredictable — arguably one of his finest turns.\n- Samantha provides the emotional centre, while Jagapathi Babu delivers the menace that drives the film’s injustices.\n- Technically superb: R. Rathnavelu’s ochre-drenched cinematography, authentic production design and Devi Sri Prasad’s woven-in score all serve the story, not the other way around.\n\nNotes for creators and leaders:\n- World-building and patience pay off — allow scenes (and characters) to breathe.\n- Casting against type can reveal new strengths.\n- Craft cohesion (direction, cinematography, sound, performance) elevates impact.\n\nCaveats: the film is l